# ROGII - Kaggle Inference A2a DWT (Internet OFF)

Offline inference for A2a DWT baseline. Uses mounted datasets:
- `rogii-repo-v2` (source code)
- `rogii-models-a2a-dwt` (trained LightGBM model)
- `rogii-wheels-a2a-dwt` (pywavelets offline dependency)
- Competition data (auto-attached)

No git clone, no pip from internet, no training.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_PATH = Path('/kaggle/working/submission.csv')
REPO_DATASET_SLUG = 'rogii-repo-v2'
MODEL_DATASET_SLUG = 'rogii-models-a2a-dwt'
WHEELS_DATASET_SLUG = 'rogii-wheels-a2a-dwt'
MODEL_FILE = 'a2a_dwt.pkl'
COMPETITION_SLUG = 'rogii-wellbore-geology-prediction'

def _path_score(path, preferred_slug):
    text = str(path).replace('\\', '/').lower()
    return (0 if preferred_slug.lower() in text else 1, len(path.parts), text)

def find_wheels_dir(input_root):
    candidates = []
    for current, dirs, files in os.walk(input_root):
        path = Path(current)
        if any(f.endswith('.whl') for f in files):
            candidates.append(path)
    if not candidates:
        raise FileNotFoundError(f'Could not find wheels; attach {WHEELS_DATASET_SLUG}')
    return sorted(candidates, key=lambda p: _path_score(p, WHEELS_DATASET_SLUG))[0]

def find_repo_root(input_root):
    candidates = []
    for current, dirs, _ in os.walk(input_root):
        dirs[:] = [d for d in dirs if d not in {'.git', '__pycache__', '.ipynb_checkpoints'}]
        path = Path(current)
        if (path / 'scripts' / 'run_predict.py').is_file() and (path / 'src' / 'rogii').is_dir():
            candidates.append(path)
    if not candidates:
        raise FileNotFoundError(f'Could not find repo; attach {REPO_DATASET_SLUG}')
    return sorted(candidates, key=lambda p: _path_score(p, REPO_DATASET_SLUG))[0]

def find_model_path(input_root):
    candidates = [p for p in input_root.rglob(MODEL_FILE) if p.is_file()]
    if not candidates:
        raise FileNotFoundError(f'Could not find {MODEL_FILE}; attach {MODEL_DATASET_SLUG}')
    return sorted(candidates, key=lambda p: _path_score(p, MODEL_DATASET_SLUG))[0]

def find_data_dir(input_root):
    candidates = []
    for current, dirs, files in os.walk(input_root):
        path = Path(current)
        if 'sample_submission.csv' in files and 'test' in dirs:
            candidates.append(path)
    if not candidates:
        raise FileNotFoundError(f'Could not find competition data; attach {COMPETITION_SLUG}')
    return sorted(candidates, key=lambda p: _path_score(p, COMPETITION_SLUG))[0]

# 1. Install pywavelets from offline wheels
wheels_dir = find_wheels_dir(INPUT_ROOT)
print('Wheels dir:', wheels_dir)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--no-index',
    '--find-links', str(wheels_dir), 'pywavelets'
], check=True)
print('pywavelets installed successfully')

# 2. Discover paths
repo_root = find_repo_root(INPUT_ROOT)
model_path = find_model_path(INPUT_ROOT)
data_dir = find_data_dir(INPUT_ROOT)

print('Repo root:', repo_root)
print('Model path:', model_path)
print('Data dir:', data_dir)
print('Output path:', OUTPUT_PATH)

# 3. Run prediction (v2 payload auto-detects feature flags)
cmd = [
    sys.executable,
    str(repo_root / 'scripts' / 'run_predict.py'),
    '--data-dir', str(data_dir),
    '--model', str(model_path),
    '--output', str(OUTPUT_PATH),
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=repo_root, check=True)

# 4. Validate output
if not OUTPUT_PATH.is_file():
    raise FileNotFoundError(f'Missing expected submission file: {OUTPUT_PATH}')
if OUTPUT_PATH.stat().st_size <= 0:
    raise ValueError(f'Empty submission file: {OUTPUT_PATH}')
print(f'Submission ready: {OUTPUT_PATH} ({OUTPUT_PATH.stat().st_size} bytes)')
